# Bio-Inspired Resource-Constrained Potential Games

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omega2417/bnt/blob/claude/publication-zenodo-project-w2iwa0/bircpg/notebooks/bircpg_colab.ipynb)

**Companion notebook to** *Bio-Inspired Resource-Constrained Potential Games for
Adaptive Coordination in Autonomous Multi-Agent Systems*.

Source: <https://github.com/omega2417/bnt/tree/claude/publication-zenodo-project-w2iwa0/bircpg>

This notebook runs the manuscript's mathematics. It reproduces Table 2, the
exact-potential identity, the approximate-equilibrium certificate, the stationary
law, the price of anarchy, Equation (15) and Figure 3 — then runs a small
Level II study with the baselines of Section 7.3.

---

## Read this before running anything

The manuscript presents a candidate formulation and an evaluation plan, **not
results**. This notebook keeps to that boundary:

| | |
|---|---|
| **Level I** — finite-game identities, equilibrium calculations | exact rational arithmetic; residuals are identically zero |
| **Level II** — synthetic environments | a harness with declared generative assumptions; synthetic task success is *not* robot validation |
| **Level III** — trained controllers, hardware, measured energy and latency | **not implemented, not claimed** |

* No experiment reported in the manuscript has been executed.
* The Level II study below runs at **demonstration scale** and supports no
  confirmatory claim. The preregistered design of Section 7.2 is available as
  `preregistered_design()` and takes hours, not minutes.
* Energy here is a **modelled proxy**. Equation (2) is a preference model, not a
  physical-energy accounting identity.
* The proposed method is **not** tuned to win. Where it loses to a baseline, the
  notebook says so.

## 0. Setup

Runs in Google Colab or locally. Takes a few seconds.

In [ ]:
# --- Setup: make `bircpg` importable in Colab or locally -------------------
import importlib, os, subprocess, sys, pathlib

# Where the package lives. In Colab this cell installs it from here directly.
REPO_URL = "https://github.com/omega2417/bnt.git"
REPO_REF = "claude/publication-zenodo-project-w2iwa0"   # branch, tag or commit
SUBDIR   = "bircpg"                                     # package root inside the repo

def _ensure_bircpg():
    try:
        importlib.import_module("bircpg")
        return "already importable"
    except ImportError:
        pass
    # Running from a checkout of the project?
    here = pathlib.Path.cwd()
    for base in [here, *here.parents][:4]:
        candidate = base / "bircpg" / "src"
        if (candidate / "bircpg" / "__init__.py").exists():
            sys.path.insert(0, str(candidate))
            return f"using local source at {candidate}"
        if (base / "src" / "bircpg" / "__init__.py").exists():
            sys.path.insert(0, str(base / "src"))
            return f"using local source at {base / 'src'}"
    if REPO_URL:
        spec = f"git+{REPO_URL}@{REPO_REF}#subdirectory={SUBDIR}"
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", spec], check=True)
        return f"installed from {REPO_URL} @ {REPO_REF}"
    # Colab with the archive to hand: upload bircpg-*.zip when prompted.
    try:
        from google.colab import files  # type: ignore
    except ImportError:
        raise ImportError(
            "bircpg not found. Run this notebook from the project directory, "
            "set REPO_URL above, or install the package with `pip install -e .`"
        )
    print("Upload the bircpg archive (bircpg-1.0.0.zip) when prompted…")
    uploaded = files.upload()
    name = next(iter(uploaded))
    subprocess.run(["unzip", "-q", "-o", name], check=True)
    root = pathlib.Path(name).stem
    sys.path.insert(0, str(pathlib.Path(root) / "src"))
    return f"unpacked {name}"

print(_ensure_bircpg())

try:
    import matplotlib, pandas  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "matplotlib", "pandas"], check=True)

import bircpg
print("bircpg", bircpg.__version__)

## 1. Section 6 — the constructed two-agent example, exactly

Two agents, two tasks, one controller mode, no congestion. $V_A = 8$, $V_B = 6$,
private costs $d_1(A)=1$, $d_1(B)=2$, $d_2(A)=2$, $d_2(B)=1$.

Every number below is a `Fraction`, not a float. This is an exact arithmetic
illustration — not a simulation, not a measurement.

In [ ]:
import pandas as pd
from bircpg.section6 import build_game, table2, summary

pd.DataFrame(table2()).rename(columns={
    "profile": "Profile", "u_1": "u₁", "u_2": "u₂",
    "welfare": "W(a)", "potential": "Φ(a)", "nash_gap": "Nash gap",
    "is_pne": "Pure NE",
})

This is Table 2 of the manuscript. The two pure equilibria have welfare **12**
and **10** — so an equilibrium certificate alone does not identify the best
allocation.

In [ ]:
s = summary()
print("Pure Nash equilibria:      ", ", ".join(s["pure_equilibria"]))
print("Welfare optimum W*:        ", s["welfare_optimum"])
print("Equilibrium welfare:       ", [str(w) for w in s["equilibrium_welfare"]])
print("Additive welfare loss:     ", s["additive_welfare_loss"], "utility units")
print("Price of anarchy (this instance only):",
      s["price_of_anarchy"], "=", float(s["price_of_anarchy"]))
print()
print("The ratio is a property of this instance. It is not a universal bound for")
print("the model class, and it is undefined when equilibrium welfare is not positive.")

## 2. Proposition 1 — the exact potential

$$\Phi(b_i,a_{-i}) - \Phi(a_i,a_{-i}) = u_i(b_i,a_{-i}) - u_i(a_i,a_{-i})$$

Checked on **every** feasible unilateral deviation from **every** feasible
profile. With rational arithmetic the residual is identically zero — there is no
tolerance to hide behind.

In [ ]:
from bircpg.equilibria import verify_exact_potential

game = build_game()
check_active = verify_exact_potential(game, active_only=True)
check_full = verify_exact_potential(game, active_only=False)

print("Between active profiles: ", check_active)
print("Including entry and exit:", check_full)
print()
print("The 8 nontrivial changes between active profiles are the ones quoted in")
print("Section 6; the full sweep also covers entry from and exit to the fallback.")

### Corollary 1 — existence and finite improvement

A maximiser of $\Phi$ exists because $\mathcal{A}$ is finite, and it is a pure
Nash equilibrium. Every strictly improving unilateral path is finite because
$\Phi$ strictly increases, so no profile can recur.

Note what the code does **not** claim: terminating a finite budget of updates
does not certify equilibrium. `improvement_path` reports `certified` only after a
complete unchanged-context sweep found nothing to improve.

In [ ]:
from bircpg.equilibria import improvement_path, potential_maximiser
from bircpg.game import OUTSIDE
from bircpg.level1 import random_instance

instance = random_instance(n_agents=5, n_tasks=3, seed=17, mask_probability=0.2)
start = tuple(OUTSIDE for _ in range(instance.n_agents))

path = improvement_path(instance, start)
print(f"accepted switches: {path.length}   sweeps: {path.sweeps}   "
      f"terminated: {path.terminated}")
print("potential along the path:", [str(p) for p in path.potentials])
print("strictly increasing:", all(b > a for a, b in zip(path.potentials, path.potentials[1:])))
print("no profile recurs:   ", len(set(path.profiles)) == len(path.profiles))
print(f"crude bound |A| - 1 = {instance.profile_count - 1} (finite, NOT a polynomial-time claim)")

budgeted = improvement_path(instance, start, max_switches=1)
print()
print(f"with a 1-switch budget: terminated={budgeted.terminated}, "
      f"certified={budgeted.certified}  ← a budget-limited exit certifies nothing")

## 3. Figure 3 — regenerated from the code

Panel (a): welfare, potential and the unilateral Nash gap for the four active
profiles. Panel (b): the Equation (12) stationary law across temperatures — as
$\tau$ falls the mass concentrates on the potential maximiser $(A,B)$; at larger
$\tau$ every profile, including the two non-equilibria, keeps appreciable
probability.

In [ ]:
import matplotlib.pyplot as plt
from bircpg.figures import figure3

figure3(theme="light", dpi=110)
plt.show()

## 4. Proposition 2 and the Equation (8) gate

Under $|\hat u_i(a) - u_i(a)| \le \delta_i$, an $\eta$-Nash equilibrium of the
*estimated* game is an $(\eta + 2\delta_{\max})$-Nash equilibrium of the true game.
No potential assumption is needed for that inequality.

For reliable improvement rather than a terminal certificate, accept a switch only
when the estimated gain exceeds $2\delta_i + \theta$. The true gain then exceeds
$\theta$, so the exact potential increases. If a complete sweep accepts nothing,
the true gap is at most $4\delta_i + \theta$.

**The factor four is the stopping certificate; the factor two is the
estimated-game equilibrium. They are different statements.**

In [ ]:
import random
from bircpg.estimation import (
    BoundedNoiseEstimator, approximate_equilibrium_certificate,
    estimated_nash_gap, uncertainty_gated_sweep,
)

delta, theta = 0.15, 0.02
estimator = BoundedNoiseEstimator(delta, instance.n_agents, rng=random.Random(0))

# Proposition 2, checked against the truth on every profile.
worst = 0.0
for profile in instance.profiles():
    eta = estimated_nash_gap(instance, profile, estimator)
    bound = approximate_equilibrium_certificate(eta, estimator.delta_max)
    worst = max(worst, float(instance.nash_gap(profile)) - bound)
print(f"Proposition 2 holds on all {instance.profile_count} profiles "
      f"(worst true gap minus bound: {worst:+.3e}, must be <= 0)")

sweep = uncertainty_gated_sweep(instance, start, estimator, theta=theta)
print()
print(f"gated sweep: {len(sweep.switches)} switches, {sweep.evaluations} payoff "
      f"evaluations, terminated={sweep.terminated}")
print(f"certified bound 4δ + θ = {sweep.true_gap_bound:.4f}")
print(f"actual true Nash gap   = {float(instance.nash_gap(sweep.profile)):.4f}  ← within the bound")

### Calibration — the part that is easy to get wrong

Assumption A5 is emphatic: a neural confidence output is **not** an error bound
without calibration or independent validation. Section 8 goes further — obtaining
that uniform allowance on a changing perception distribution can be harder than
deriving the equilibrium bound itself.

So the package reports the *empirical coverage rate* and the *declared bound*
separately. They are not the same thing, and an empirical rate never upgrades
into a deterministic guarantee.

In [ ]:
from bircpg.estimation import coverage_report

report = coverage_report(instance, estimator, list(instance.profiles()))
for key, value in report.items():
    print(f"{key:>22}: {value}")

## 5. Proposition 3 — the stationary law, checked independently

The closed form of Equation (12) is

$$\pi_\tau(a) = Z^{-1}\exp\!\big(\Phi(a)/\tau\big)\prod_i q_i(a_i\mid z)$$

It is verified against an **independently constructed** transition matrix:
stochasticity, detailed balance, and the stationary residual, reported
separately as Section 7.1 requires.

A stationary-law check is not a finite-time mixing bound, and the joint
stationary distribution is not automatically a quantal-response or correlated
equilibrium.

In [ ]:
from bircpg.logit import (detailed_balance_residual, stationary_distribution,
                          transition_matrix)
from bircpg.traces import reference_distribution

z = [0.8, 0.2, 0.5][: instance.n_tasks]
refs = [reference_distribution(instance.action_sets[i], z, kappa=1.0, epsilon_z=0.1)
        for i in range(instance.n_agents)]

profiles, P = transition_matrix(instance, refs, tau=0.5)
pi = stationary_distribution(instance, refs, tau=0.5)
for name, residual in detailed_balance_residual(profiles, P, pi).items():
    print(f"{name:>18} residual: {residual:.3e}")
print(f"{'probabilities sum':>18}: {sum(pi.values()):.12f}")

### Equation (15) — the manuscript's published probabilities

In [ ]:
from bircpg.section6 import equation15

published = {"(A, A)": 0.04192, "(A, B)": 0.84203, "(B, A)": 0.11396, "(B, B)": 0.00209}
computed = equation15(tau=1.0)
pd.DataFrame({
    "computed here": pd.Series(computed).round(5),
    "manuscript Eq (15)": pd.Series(published),
})

Note what Equation (15) shows: **a stationary logit distribution is not a point
equilibrium.** $(A,A)$ and $(B,B)$ are not Nash equilibria, and both keep
positive probability at $\tau = 1$. It establishes neither fast mixing nor
improved energy efficiency.

### Corollary 2 — the potential-loss bound

$$\max_a \Phi(a) - \mathbb{E}_{\pi_\tau}[\Phi(a)] \;\le\; \tau\big(\log K + R_q\big)$$

Conservative, and about the *potential*, not welfare. Very concentrated
references (large $R_q$) weaken it — visible below.

In [ ]:
from bircpg.logit import potential_loss_bound

rows = []
for kappa, eps, label in [(0.0, 1.0, "uniform"), (1.0, 0.1, "mild trace bias"),
                          (8.0, 0.001, "sharply concentrated")]:
    r = [reference_distribution(instance.action_sets[i], z, kappa, eps)
         for i in range(instance.n_agents)]
    for tau in (0.1, 0.5, 2.0):
        b = potential_loss_bound(instance, r, tau)
        rows.append({"references": label, "τ": tau, "R_q": round(b["R_q"], 3),
                     "realised loss": round(b["realised_loss"], 4),
                     "bound τ(log K + R_q)": round(b["bound"], 4),
                     "holds": b["satisfied"]})
pd.DataFrame(rows)

## 6. Level I — the Section 7.1 verification sweep

Seeded exact instances plus the edge cases the manuscript asks for: ties, an
agent whose only feasible action is the fallback, a zero-value task, and
heterogeneous resource masks.

In [ ]:
from bircpg.level1 import level1_sweep

reports = level1_sweep(agent_counts=(2, 3, 4), task_counts=(2, 3), seeds=10, chain_tau=0.5)
chains = [r.chain for r in reports if r.chain]

print(f"instances: {len(reports)}   profiles enumerated: {sum(r.profiles for r in reports)}")
print(f"Equation (6) residual identically zero: {sum(r.potential_exact for r in reports)}/{len(reports)}")
print(f"potential maximiser is a pure NE:       {sum(r.potential_maximiser_is_pne for r in reports)}/{len(reports)}")
print(f"Corollary 2 bound satisfied:            {sum(c['satisfied'] for c in chains)}/{len(chains)}")
print(f"max detailed-balance residual:          {max(c['detailed_balance'] for c in chains):.3e}")

losses = [float(r.additive_welfare_loss) for r in reports if r.additive_welfare_loss is not None]
print(f"\nadditive welfare loss at the worst pure equilibrium: "
      f"mean {sum(losses)/len(losses):.4f}, max {max(losses):.4f}")
print("Decentralised equilibrium and collective efficiency are distinct — that gap")
print("is the quantity RQ5 asks about, and it is measured, not bounded a priori.")

## 7. Level II — a small synthetic study

Abstract monitoring sites on a unit square, distance-based travel costs,
congestion $g_j(k) = \gamma_j(k-1)$, against the baselines of Section 7.3.

**This is demonstration scale.** It is not the preregistered design and supports
no confirmatory claim. Read the numbers as "the harness runs and the comparators
behave as specified", nothing more.

One detail worth knowing: Section 7.2's change schedule (25% of task values every
100 epochs, 10% of agents removed at epoch 500) is written for a 1000-epoch
horizon. Below that horizon it would simply never fire, and a "changing values"
condition in which nothing changes would silently duplicate the stationary one.
So the schedule is scaled proportionally and the condition is relabelled
`+scaled`, which is why that suffix appears below.

In [ ]:
from bircpg.study import StudyDesign, run_study
from bircpg.metrics import endpoints

design = StudyDesign(
    population_sizes=(12,),
    conditions=("stationary_nominal", "noisy_sensing", "changing_values"),
    seeds=6,
    epochs=40,
)
print(f"{design.runs_per_method} runs per method × 7 methods")
results = run_study(design)

rows = [endpoints(r, design.gap_tolerance).as_dict() for r in results]
frame = pd.DataFrame(rows)
summary_table = frame.groupby("method").agg(
    terminal_gap=("terminal_nash_gap", "mean"),
    cumulative_welfare=("cumulative_welfare", "mean"),
    payoff_evaluations=("payoff_evaluations", "mean"),
    inference_energy=("inference_energy", "mean"),
    certified=("certified_fraction", "mean"),
    oracle=("uses_true_payoffs", "max"),
).round(3).sort_values("cumulative_welfare", ascending=False)
summary_table

Read that table with the labels in mind:

* `oracle = True` marks an **informational advantage** (B4 gets correct payoff
  differences; B5 enumerates welfare centrally). They are yardsticks, not
  deployable low-communication methods under packet loss.
* `certified` is the fraction of epochs that ended with a genuine terminal
  certificate rather than a budget-limited exit. The logit branch never
  certifies — it deliberately permits payoff-decreasing moves.
* `inference_energy` is a **modelled proxy** from the declared synthetic
  profile. It is not a measured watt-hour and cannot be reported as one.

### The prespecified paired comparison

H1 predicts that in slowly varying environments informative traces reduce the
payoff evaluations needed to reach a Nash-gap threshold, relative to the
otherwise identical uniform-reference rule (B2). H2 predicts that delayed or
misleading traces can reverse that advantage under rapid change.

Pairing is on identical environmental seeds; the bootstrap resamples **whole
runs**, because epochs within a run are dependent observations. Censored runs —
those that never reached the tolerance — are dropped and counted, never replaced
with a favourable value.

In [ ]:
import random
from bircpg.study import PRIMARY_OUTCOMES, paired_comparison
from bircpg.metrics import holm_adjust

rng = random.Random(0)
for condition in sorted({r.condition for r in results}):
    result = paired_comparison(
        results, "proposed-logit", "B2-uniform-logit",
        outcome="cumulative_welfare", condition=condition,
        resamples=10_000, rng=rng,
    )
    verdict = "excludes zero" if result.excludes_zero else "includes zero"
    print(f"{condition:32s} mean diff {result.mean_difference:+8.3f} "
          f"[{result.ci_low:+7.3f}, {result.ci_high:+7.3f}]  n={result.n_pairs}  {verdict}")

print()
print("Positive = the trace-biased rule did better. In this demonstration setting")
print("it does NOT, in any condition. That result is left exactly as it came out:")
print("Section 8 is explicit that a trace can worsen adaptation, and the generator")
print("is built so that no method is handed a cheaper cost or a better accuracy.")
print()
print("A confirmatory reading would need the preregistered design, the prespecified")
print("primary outcome per hypothesis, and multiplicity correction:")
for row in holm_adjust([0.01, 0.04, 0.2], list(PRIMARY_OUTCOMES)):
    print(f"  {row['label']}: p_raw={row['p_raw']:.3f}  p_Holm={row['p_holm']:.3f}   "
          f"({PRIMARY_OUTCOMES[row['label']]['outcome']})")
print("  (illustrative p-values — no test was performed here)")

## 8. Running the preregistered design

Section 7.2's core design: 4 population sizes × 6 prespecified conditions × 30
environmental seeds × 1000 epochs = **720 seed-condition runs per method**.

It is an initial design size, not a claim of adequate statistical power; power or
precision should be reassessed from an independent pilot, and the configuration
should be frozen *before* the evaluation seeds are opened.

```python
from bircpg.study import preregistered_design, run_study, write_endpoints_csv

design  = preregistered_design()      # hours, not minutes
results = run_study(design)
write_endpoints_csv(results, "results/level2_endpoints.csv")
```

## What this notebook established, and what it did not

**Established** — by computation, reproducibly:

* Table 2, the price of anarchy $6/5$, and Equation (15) match the manuscript exactly.
* The Equation (6) potential identity holds with an identically zero residual.
* A potential maximiser is a pure Nash equilibrium; improvement paths terminate.
* Proposition 2's certificate and the Equation (8) gate hold against ground truth.
* Proposition 3's stationary law satisfies detailed balance; Corollary 2's bound holds.
* Decentralised equilibrium and collective efficiency are distinct quantities.

**Not established, and not claimed:**

* Any energy saving. Energy here is a modelled proxy, never a measurement.
* Any coordination advantage for the proposed method — in the demonstration
  condition above it loses to its own uniform-reference comparator.
* Any robotic or biological result. Level III is not implemented.
* Any finite-time convergence guarantee. A stationary law is not a mixing bound.

Claims of energy savings, superior coordination or robust real-world autonomy
remain hypotheses until the corresponding evaluation is completed.